In [ ]:
!pip -q install -U \
  "transformers==4.41.2" \
  "tokenizers==0.19.1" \
  "datasets==2.20.0" \
  "accelerate==0.33.0" \
  "peft==0.12.0" \
  sentencepiece

In [1]:
import os, json, re, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import transformers, tokenizers, datasets, accelerate, peft
print("transformers", transformers.__version__)
print("tokenizers", tokenizers.__version__)
print("datasets", datasets.__version__)
print("accelerate", accelerate.__version__)
print("peft", peft.__version__)

CUDA: True
GPU: Tesla P100-PCIE-16GB
transformers 4.41.2
tokenizers 0.19.1
datasets 2.20.0
accelerate 0.33.0
peft 0.12.0


In [2]:
from pathlib import Path

MODEL_NAME = "Salesforce/codet5-base"

TRAIN_V1 = "/kaggle/input/datasets/mahithgangu/cd-patch-dataset-v1/train.jsonl"
VAL_V1   = "/kaggle/input/datasets/mahithgangu/cd-patch-dataset-v1/val.jsonl"

WORK = Path("/kaggle/working")
DATA_V2 = WORK / "patch_dataset_v2"
DATA_V2.mkdir(parents=True, exist_ok=True)

TRAIN_V2 = DATA_V2 / "train.jsonl"
VAL_V2   = DATA_V2 / "val.jsonl"

OUT_DIR = WORK / "outputs_v2"
RUN_DIR = WORK / "runs_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_V1:", TRAIN_V1)
print("VAL_V1  :", VAL_V1)
print("TRAIN_V2:", TRAIN_V2)
print("VAL_V2  :", VAL_V2)
print("OUT_DIR :", OUT_DIR)
print("RUN_DIR :", RUN_DIR)

TRAIN_V1: /kaggle/input/datasets/mahithgangu/cd-patch-dataset-v1/train.jsonl
VAL_V1  : /kaggle/input/datasets/mahithgangu/cd-patch-dataset-v1/val.jsonl
TRAIN_V2: /kaggle/working/patch_dataset_v2/train.jsonl
VAL_V2  : /kaggle/working/patch_dataset_v2/val.jsonl
OUT_DIR : /kaggle/working/outputs_v2
RUN_DIR : /kaggle/working/runs_v2


In [3]:
cmd_re = re.compile(r"^\s*(DEL|INS|REP)\b(.*)$")

def make_target_v2(row):
    t = row["target"].strip()
    m = cmd_re.match(t)
    if not m:
        return f"UNK_LC {row['error_line']} {row['error_col']}"
    cmd = m.group(1).upper()
    rest = m.group(2).strip()

    toks = rest.split()
    j = 0
    while j < len(toks) and toks[j].isdigit():
        j += 1
    payload = " ".join(toks[j:]).strip()

    base = f"{cmd}_LC {row['error_line']} {row['error_col']}"
    return base + (" " + payload if payload else "")

def convert_file(in_path, out_path):
    n = 0
    with open(in_path, "r", encoding="utf-8") as fin, open(out_path, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            row["target"] = make_target_v2(row)
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            n += 1
    return n

n_train = convert_file(TRAIN_V1, TRAIN_V2)
n_val   = convert_file(VAL_V1, VAL_V2)

print("Wrote v2 train:", TRAIN_V2, "rows:", n_train)
print("Wrote v2 val  :", VAL_V2, "rows:", n_val)

# show a few
with open(TRAIN_V2, "r", encoding="utf-8") as f:
    for _ in range(5):
        r = json.loads(next(f))
        print("family:", r.get("family"), "| target:", r["target"])

Wrote v2 train: /kaggle/working/patch_dataset_v2/train.jsonl rows: 74689
Wrote v2 val  : /kaggle/working/patch_dataset_v2/val.jsonl rows: 2590
family: ternary_extra_colon | target: DEL_LC 145 23
family: ternary_extra_colon | target: DEL_LC 96 23
family: ternary_extra_colon | target: DEL_LC 276 23
family: ternary_extra_colon | target: DEL_LC 96 23
family: ternary_extra_colon | target: DEL_LC 94 23


In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw = load_dataset("json", data_files={"train": str(TRAIN_V2), "validation": str(VAL_V2)})
print(raw)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['task', 'split_hint', 'family', 'error_line', 'error_col', 'input', 'target'],
        num_rows: 74689
    })
    validation: Dataset({
        features: ['task', 'split_hint', 'family', 'error_line', 'error_col', 'input', 'target'],
        num_rows: 2590
    })
})


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

In [5]:
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 32

def preprocess(batch):
    inputs = [
        f"PATCH\nLINE {ln}\nCOL {col}\n\n{inp}"
        for ln, col, inp in zip(batch["error_line"], batch["error_col"], batch["input"])
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_SOURCE_LEN,
        truncation=True,
        padding=False,
    )

    labels = tokenizer(
        batch["target"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tok = raw.map(preprocess, batched=True, remove_columns=raw["train"].column_names)
print(tok)
print("Example lens:", len(tok["train"][0]["input_ids"]), len(tok["train"][0]["labels"]))

Map:   0%|          | 0/74689 [00:00<?, ? examples/s]

Map:   0%|          | 0/2590 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 74689
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2590
    })
})
Example lens: 130 8


In [6]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q", "k", "v", "o"],
)

model = get_peft_model(base_model, lora)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

trainable params: 3,538,944 || all params: 226,420,992 || trainable%: 1.5630


In [7]:
def norm_cmd(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"\s+", " ", s)
    return s.upper()

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_text = tokenizer.batch_decode(preds, skip_special_tokens=True)
    gold_text = tokenizer.batch_decode(labels, skip_special_tokens=True)

    pred_text = [norm_cmd(p) for p in pred_text]
    gold_text = [norm_cmd(g) for g in gold_text]

    exact = float(np.mean([p == g for p, g in zip(pred_text, gold_text)]))
    return {"exact_match": exact}

In [8]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 3e-4
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
EPOCHS_FULL = 2

args_full = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR),
    eval_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    logging_steps=50,
    save_total_limit=2,

    num_train_epochs=EPOCHS_FULL,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,

    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    generation_num_beams=1,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=args_full,
    train_dataset=tok["train"],
    eval_dataset=tok["validation"],
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer (full) ready:", len(trainer.train_dataset), len(trainer.eval_dataset))

# Save config for your report
with open(RUN_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "train_v1": TRAIN_V1,
        "val_v1": VAL_V1,
        "train_v2": str(TRAIN_V2),
        "val_v2": str(VAL_V2),
        "max_source_len": MAX_SOURCE_LEN,
        "max_target_len": MAX_TARGET_LEN,
        "batch_size": BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "epochs_full": EPOCHS_FULL,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
    }, f, indent=2)

print("Wrote:", RUN_DIR / "config.json")

2026-03-05 09:10:11.997473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772701812.169358     115 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772701812.218369     115 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772701812.637226     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772701812.637270     115 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772701812.637273     115 computation_placer.cc:177] computation placer alr

Trainer (full) ready: 74689 2590
Wrote: /kaggle/working/runs_v2/config.json


In [9]:
trainer.train()

Step,Training Loss,Validation Loss,Exact Match
1000,0.005700,0.000773,0.999614
2000,0.000800,0.000034,1.000000
3000,0.000600,0.000017,1.000000
4000,0.001100,0.000403,1.000000
5000,0.000100,0.000013,1.000000
6000,0.000100,0.000010,1.000000
7000,0.000100,0.000004,1.000000
8000,0.000100,0.000003,1.000000
9000,0.000000,0.000002,1.000000


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in ver

TrainOutput(global_step=9336, training_loss=0.10332300984981009, metrics={'train_runtime': 4709.989, 'train_samples_per_second': 31.715, 'train_steps_per_second': 1.982, 'total_flos': 2.314179903446323e+16, 'train_loss': 0.10332300984981009, 'epoch': 1.9997857984363285})

In [10]:
import os, json, time

# Final eval on full validation set
metrics = trainer.evaluate()
print(metrics)

# Save eval metrics for your Week 12/13 report
eval_path = os.path.join(RUN_DIR, "eval_metrics.json")
with open(eval_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("✅ Wrote:", eval_path)

{'eval_loss': 2.293800434927107e-06, 'eval_exact_match': 1.0, 'eval_runtime': 98.2086, 'eval_samples_per_second': 26.372, 'eval_steps_per_second': 3.299, 'epoch': 1.9997857984363285}
✅ Wrote: /kaggle/working/runs_v2/eval_metrics.json


In [11]:
import os

adapter_dir = os.path.join(OUT_DIR, "adapter")
os.makedirs(adapter_dir, exist_ok=True)

trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

print("✅ Saved adapter+tokenizer to:", adapter_dir)
print("Files:", os.listdir(adapter_dir))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Saved adapter+tokenizer to: /kaggle/working/outputs_v2/adapter
Files: ['vocab.json', 'tokenizer.json', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'merges.txt', 'tokenizer_config.json', 'special_tokens_map.json']


In [12]:
# Saves trainer state + any extra files (optional but good)
trainer.save_state()
print("✅ Trainer state saved in:", OUT_DIR)

✅ Trainer state saved in: /kaggle/working/outputs_v2


In [14]:
TRAIN_PATH = "/kaggle/working/patch_dataset_v2/train.jsonl"
VAL_PATH   = "/kaggle/working/patch_dataset_v2/val.jsonl"

print("TRAIN_PATH:", TRAIN_PATH)
print("VAL_PATH  :", VAL_PATH)
print("OUT_DIR   :", OUT_DIR)
print("RUN_DIR   :", RUN_DIR)

TRAIN_PATH: /kaggle/working/patch_dataset_v2/train.jsonl
VAL_PATH  : /kaggle/working/patch_dataset_v2/val.jsonl
OUT_DIR   : /kaggle/working/outputs_v2
RUN_DIR   : /kaggle/working/runs_v2


In [15]:
import json, re, os
from datasets import load_dataset

def norm_cmd(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s.upper()

N_SAMPLES = 200

val_raw = load_dataset("json", data_files={"validation": VAL_PATH})["validation"].select(range(N_SAMPLES))
val_tok = val_raw.map(preprocess, batched=True, remove_columns=val_raw.column_names)

preds = trainer.predict(val_tok, max_length=MAX_TARGET_LEN, num_beams=1)
pred_text = tokenizer.batch_decode(preds.predictions, skip_special_tokens=True)

samples_path = os.path.join(RUN_DIR, f"val_samples_{N_SAMPLES}.jsonl")

exact = 0
with open(samples_path, "w", encoding="utf-8") as f:
    for i in range(len(val_raw)):
        gold = val_raw[i]["target"].strip()
        pred = pred_text[i].strip()
        is_exact = (norm_cmd(gold) == norm_cmd(pred))
        exact += int(is_exact)

        f.write(json.dumps({
            "family": val_raw[i].get("family"),
            "error_line": val_raw[i].get("error_line"),
            "error_col": val_raw[i].get("error_col"),
            "gold": gold,
            "pred": pred,
            "exact": is_exact,
        }, ensure_ascii=False) + "\n")

print("✅ Wrote:", samples_path)
print("Exact-match on samples:", exact / N_SAMPLES)

Generating validation split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Wrote: /kaggle/working/runs_v2/val_samples_200.jsonl
Exact-match on samples: 0.73


In [16]:
import json, os
from collections import Counter
from datasets import load_dataset

raw_all = load_dataset("json", data_files={"train": TRAIN_PATH, "validation": VAL_PATH})

train_fam = Counter(raw_all["train"]["family"])
val_fam   = Counter(raw_all["validation"]["family"])

stats = {
    "train_rows": len(raw_all["train"]),
    "val_rows": len(raw_all["validation"]),
    "unique_families_train": len(train_fam),
    "unique_families_val": len(val_fam),
    "top10_train_families": train_fam.most_common(10),
    "top10_val_families": val_fam.most_common(10),
}

stats_path = os.path.join(RUN_DIR, "dataset_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2)

print("✅ Wrote:", stats_path)
print(stats)

✅ Wrote: /kaggle/working/runs_v2/dataset_stats.json
{'train_rows': 74689, 'val_rows': 2590, 'unique_families_train': 35, 'unique_families_val': 31, 'top10_train_families': [('broken_logical_op', 2707), ('missing_rparen', 2682), ('for_missing_first_semicolon', 2660), ('for_missing_second_semicolon', 2652), ('missing_lparen_general', 2652), ('extra_rbrace', 2626), ('missing_comma_args', 2585), ('missing_lbrace', 2554), ('mismatch_rparen_to_rbrack', 2543), ('missing_operand', 2525)], 'top10_val_families': [('extra_operator', 260), ('compound_assign_split', 234), ('extra_rparen', 219), ('missing_rbrace', 204), ('missing_lbrace', 189), ('missing_semicolon', 150), ('illegal_char', 118), ('mismatch_lparen_to_lbrack', 116), ('missing_lparen_control', 110), ('extra_rbrack', 104)]}


In [17]:
import json, os, re
from collections import defaultdict
from datasets import load_dataset

def norm_cmd(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s.upper()

val_full = load_dataset("json", data_files={"validation": VAL_PATH})["validation"]
val_full_tok = val_full.map(preprocess, batched=True, remove_columns=val_full.column_names)

preds = trainer.predict(val_full_tok, max_length=MAX_TARGET_LEN, num_beams=1)
pred_text = tokenizer.batch_decode(preds.predictions, skip_special_tokens=True)

by_family = defaultdict(lambda: {"n": 0, "correct": 0})

for i in range(len(val_full)):
    fam = val_full[i].get("family", "UNKNOWN")
    gold = norm_cmd(val_full[i]["target"])
    pred = norm_cmd(pred_text[i])
    by_family[fam]["n"] += 1
    by_family[fam]["correct"] += int(pred == gold)

family_acc = {k: v["correct"] / max(1, v["n"]) for k, v in by_family.items()}
family_acc_sorted_worst = sorted(family_acc.items(), key=lambda x: x[1])

out = {
    "overall_exact": sum(v["correct"] for v in by_family.values()) / max(1, sum(v["n"] for v in by_family.values())),
    "worst_30_families": family_acc_sorted_worst[:30],
    "best_30_families": sorted(family_acc.items(), key=lambda x: x[1], reverse=True)[:30],
}

family_path = os.path.join(RUN_DIR, "family_accuracy.json")
with open(family_path, "w", encoding="utf-8") as f:
    json.dump(out, f, indent=2)

print("✅ Wrote:", family_path)
print("Overall exact:", out["overall_exact"])
print("Worst 10:", out["worst_30_families"][:10])

Map:   0%|          | 0/2590 [00:00<?, ? examples/s]

✅ Wrote: /kaggle/working/runs_v2/family_accuracy.json
Overall exact: 0.9382239382239382
Worst 10: [('missing_comma_args', 0.0), ('missing_comma_params', 0.0), ('missing_lbrack', 1.0), ('mismatch_rbrack_to_rparen', 1.0), ('extra_operator', 1.0), ('illegal_char', 1.0), ('missing_semicolon', 1.0), ('compound_assign_split', 1.0), ('missing_lbrace', 1.0), ('missing_rbrack', 1.0)]


In [18]:
!zip -r /kaggle/working/runs_v2.zip /kaggle/working/runs_v2 > /dev/null
!zip -r /kaggle/working/adapter_v2.zip /kaggle/working/outputs_v2/adapter > /dev/null
!ls -lah /kaggle/working/*.zip

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


-rw-r--r-- 1 root root  14M Mar  5 11:08 /kaggle/working/adapter_v2.zip
-rw-r--r-- 1 root root 3.8K Mar  5 11:08 /kaggle/working/runs_v2.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
!zip -r /kaggle/working/patch_dataset_v2.zip /kaggle/working/patch_dataset_v2 > /dev/null
!ls -lah /kaggle/working/patch_dataset_v2.zip

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


-rw-r--r-- 1 root root 722K Mar  5 11:08 /kaggle/working/patch_dataset_v2.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [20]:
!ls -R /kaggle/working/outputs_v2

/kaggle/working/outputs_v2:
adapter  checkpoint-8000  checkpoint-9000  trainer_state.json

/kaggle/working/outputs_v2/adapter:
adapter_config.json	   merges.txt  special_tokens_map.json	tokenizer.json
adapter_model.safetensors  README.md   tokenizer_config.json	vocab.json

/kaggle/working/outputs_v2/checkpoint-8000:
adapter_config.json	   rng_state.pth	    trainer_state.json
adapter_model.safetensors  scheduler.pt		    training_args.bin
merges.txt		   special_tokens_map.json  vocab.json
optimizer.pt		   tokenizer_config.json
README.md		   tokenizer.json

/kaggle/working/outputs_v2/checkpoint-9000:
adapter_config.json	   rng_state.pth	    trainer_state.json
adapter_model.safetensors  scheduler.pt		    training_args.bin
merges.txt		   special_tokens_map.json  vocab.json
optimizer.pt		   tokenizer_config.json
README.md		   tokenizer.json


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
!zip -r /kaggle/working/codet5_patch_adapter.zip /kaggle/working/outputs_v2/adapter

  adding: kaggle/working/outputs_v2/adapter/ (stored 0%)
  adding: kaggle/working/outputs_v2/adapter/vocab.json (deflated 59%)
  adding: kaggle/working/outputs_v2/adapter/tokenizer.json (deflated 72%)
  adding: kaggle/working/outputs_v2/adapter/README.md (deflated 66%)
  adding: kaggle/working/outputs_v2/adapter/adapter_config.json (deflated 52%)
  adding: kaggle/working/outputs_v2/adapter/adapter_model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: kaggle/working/outputs_v2/adapter/merges.txt (deflated 54%)
  adding: kaggle/working/outputs_v2/adapter/tokenizer_config.json (deflated 94%)
  adding: kaggle/working/outputs_v2/adapter/special_tokens_map.json (deflated 97%)


In [22]:
!ls -lh /kaggle/working | sed -n '1,200p'

total 28M
-rw-r--r-- 1 root root  14M Mar  5 11:08 adapter_v2.zip
-rw-r--r-- 1 root root  14M Mar  5 11:41 codet5_patch_adapter.zip
drwxr-xr-x 5 root root 4.0K Mar  5 10:42 outputs_v2
drwxr-xr-x 2 root root 4.0K Mar  5 09:08 patch_dataset_v2
-rw-r--r-- 1 root root 722K Mar  5 11:08 patch_dataset_v2.zip
drwxr-xr-x 2 root root 4.0K Mar  5 11:04 runs_v2
-rw-r--r-- 1 root root 3.8K Mar  5 11:08 runs_v2.zip


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
